In [ ]:
import os
from dotenv import load_dotenv
import snowflake.connector

load_dotenv()

def get_connection():
    conn = snowflake.connector.connect(
    user=os.getenv("user"),
    password=os.getenv("pass"),
    account=os.getenv("account"),
    database=os.getenv("database"),
    schema=os.getenv("schema"),
    warehouse=os.getenv("warehouse"),
    )
    return conn


clients = ["69c6b7922afeebc19cedeba5"]
conn = get_connection()
cur = conn.cursor()
  

for c in clients:
    cur.execute("""
                    select any_value(productId) from tracks t join products p on p.id = t.productId where t.userId = %s group by eventType
                    """, (c))
    data = cur.fetchall()
    print(data)

[('69c6b7922afeebc19cedeba5', '68b59a5514dd9fa395f77a55', 'click', None, 4, datetime.datetime(2026, 3, 31, 8, 10, 9, 646000), '68b59a5514dd9fa395f77a55', 'a STRAIGHT-LEG CARGO TROUSERS', 'Men', 'Bottomwear', 'STRAIGHT-LEG CARGO TROUSERS', 2000.0), ('69c6b7922afeebc19cedeba5', '69cbcf1bb5dc9158fc50bddf', 'click', None, 19, datetime.datetime(2026, 3, 31, 8, 9, 31, 659000), '69cbcf1bb5dc9158fc50bddf', 'Bonnet style bucket. Détail coutures contraste.', 'Kids', 'Topwear', 'BONNET BUCKET COUTURES', 2299.98), ('69c6b7922afeebc19cedeba5', '69cbcf1bb5dc9158fc50bddf', 'click', None, 2, datetime.datetime(2026, 3, 31, 8, 10, 19, 41000), '69cbcf1bb5dc9158fc50bddf', 'Bonnet style bucket. Détail coutures contraste.', 'Kids', 'Topwear', 'BONNET BUCKET COUTURES', 2299.98)]


In [17]:
def encode_products():
    text = []
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("select * from products")
    data = cur.fetchall()
    for d in data:
        print(d)
        text.append(d[1]+" "+d[2]+" "+d[3]+" "+d[4])
    return text

In [23]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
text = encode_products()
print(text)
vectors = model.encode(text)
print(vectors)

('68b598a314dd9fa395f77a38', 'a men sweater', 'Men', 'Topwear', 'men sweater', 5000.0)
('68b599ad14dd9fa395f77a4c', 'a REGULAR FIT TEXTURED SWEATER', 'Men', 'Topwear', 'REGULAR FIT TEXTURED SWEATER', 3200.0)
('68b59a5514dd9fa395f77a55', 'a STRAIGHT-LEG CARGO TROUSERS', 'Men', 'Bottomwear', 'STRAIGHT-LEG CARGO TROUSERS', 2000.0)
('68b59aff14dd9fa395f77a63', 'a TECHNICAL PUFFER JACKET', 'Men', 'Winterwear', 'TECHNICAL PUFFER JACKET', 6000.0)
('69cbce0cb5dc9158fc50bddd', 'Veste longue type manteau confectionnée en tissu effet daim. Col à revers crantés et manches longues. Poches à rabat sur le devant. Détail de fermoir type bijou. Fermeture avant boutonnée.', 'Women', 'Topwear', 'MANTEAU VESTE LONGUE EFFET DAIM FERMOIR', 5000.0)
('69cbcf1bb5dc9158fc50bddf', 'Bonnet style bucket. Détail coutures contraste.', 'Kids', 'Topwear', 'BONNET BUCKET COUTURES', 2299.98)
['a men sweater Men Topwear men sweater', 'a REGULAR FIT TEXTURED SWEATER Men Topwear REGULAR FIT TEXTURED SWEATER', 'a STRAIGHT-L

In [25]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)
similarity

array([[0.9999994 , 0.66470134, 0.43417785, 0.4809558 , 0.26856333,
        0.12095955],
       [0.66470134, 1.0000001 , 0.4366557 , 0.4942758 , 0.3910465 ,
        0.29230517],
       [0.43417785, 0.4366557 , 1.0000006 , 0.44156924, 0.34984753,
        0.24055935],
       [0.4809558 , 0.4942758 , 0.44156924, 1.0000002 , 0.2878429 ,
        0.22672711],
       [0.26856333, 0.3910465 , 0.34984753, 0.2878429 , 1.0000004 ,
        0.29646763],
       [0.12095955, 0.29230517, 0.24055935, 0.22672711, 0.29646763,
        0.99999994]], dtype=float32)

In [ ]:
def content_based_recommendation(product_id):
    conn = get_connection()
    cur = conn.cursor()
    cur.execute("select * from products")
    data = cur.fetchall()
    index = None
    for i in data:
        if i[0] == product_id:
            index = data.index(i)
            break
    results = [similarity[index][i] for i in range(len(similarity[index])) if i!=index and similarity[index][i]>0.6]
    print(results)
    return [i for i in results]
    
content_based_recommendation("68b598a314dd9fa395f77a38")



[1]
